This file is to experiment with setting thresholds for our data with NLP-produced similarity scores
- for now, just using some sample file(s) from get_scores()
- sectioning off data by each .1 of similarity scores (.9-1.0, .8-.9, etc)
- then, get cohen's kappas on that for comparison

In [28]:
import pandas as pd
from NLP_Eval_for_DE import scores, data
import ast
from sklearn.metrics import cohen_kappa_score, f1_score, confusion_matrix

In [29]:
testable_data = data.get_testable_data("Example\\inputs\\case study 1 input\\pain points full.csv")
codes = data.get_codes("Example\\inputs\\case study 1 input\\short titles+descriptions.csv")
all_scores = scores.get_Jina_scores(testable_data, codes)[1]

#split the lots of scores in the "Similarity scores" column into separate columns
all_scores_expanded = all_scores.copy()
all_scores_expanded[["1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11"]] = pd.DataFrame(all_scores_expanded["Similarity scores"].tolist(), index=all_scores_expanded.index)
# add the "Consensus code" column of testable_data to all_scores_expanded
all_scores_expanded["Consensus code"] = testable_data["Consensus code"].tolist()
#delete the "Similarity scores" column
all_scores_expanded = all_scores_expanded.drop(columns=["Similarity scores"])
all_scores_expanded

,Input phrase,1,2,3,4,5,6,7,8,9,10,11,Consensus code
0,cutting wood,0.655549,0.652874,0.645258,0.735443,0.682956,0.741813,0.715049,0.732674,0.694089,0.675520,0.606408,0
1,didn�t know how to use lathe,0.797545,0.706240,0.689279,0.693151,0.703598,0.788048,0.770484,0.679184,0.700546,0.664698,0.617746,1
2,Finding drill,0.711917,0.764968,0.655588,0.654544,0.723438,0.709729,0.730768,0.641555,0.825180,0.674953,0.643374,9
3,Taking out trash,0.630425,0.608344,0.646132,0.646396,0.705345,0.692943,0.664894,0.758600,0.677858,0.810726,0.589942,10
4,Finding clamp,0.731759,0.639973,0.637264,0.672274,0.833259,0.677210,0.708242,0.662600,0.712998,0.697646,0.615647,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...
394,Incorrect size gloves,0.668308,0.738858,0.797413,0.684288,0.647133,0.629817,0.651951,0.678479,0.662768,0.668407,0.615992,3
395,Uncleaned machines from previous users,0.704094,0.641190,0.673434,0.639112,0.738281,0.727613,0.722772,0.682446,0.671655,0.720414,0.622721,6
396,Machine incorrectly set up by previous user,0.727318,0.683876,0.683272,0.698322,0.746937,0.716930,0.760753,0.656647,0.702895,0.716270,0.669959,1
397,Unusable wood scarps were discarded in wrong p...,0.678698,0.660548,0.692574,0.710192,0.777346,0.762790,0.682856,0.826161,0.673755,0.738667,0.644955,8


In [30]:
def eval_filtered(all_scores_expanded, min, max):
    # now filter based on threshold 
    # drop any rows where the highest score from the 11 scores is not between min and max
    all_scores_expanded_filtered = all_scores_expanded[
        (all_scores_expanded[["1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11"]].max(axis=1) >= min) 
        & (all_scores_expanded[["1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11"]].max(axis=1) <= max)]

    # now, get cohens kappa score for the filtered data
    ground_truths = all_scores_expanded_filtered["Consensus code"].tolist()
    predictions = all_scores_expanded_filtered[["1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11"]].idxmax(axis=1).tolist()
    # convert predictions from strings to ints
    predictions = [int(x) for x in predictions]
    f1s = f1_score(ground_truths, predictions, labels=[1,2,3,4,5,6,7,8,9,10,11], average=None, zero_division=0.0) 
    mtx = confusion_matrix(ground_truths, predictions, labels=[1,2,3,4,5,6,7,8,9,10,11])
    kappa = cohen_kappa_score(ground_truths, predictions, labels=[1,2,3,4,5,6,7,8,9,10,11], weights=None, sample_weight=None)
    return [mtx, f1s, kappa]

In [31]:
rows = []
for i, j in [(0.9, 1.0), (0.8, 0.9), (0.7, 0.8), (0.6, 0.7), (0.5, 0.6), (0.4, 0.5), (0.3, 0.4), (0.2, 0.3), (0.1, 0.2), (0.0, 0.1)]:
    results = eval_filtered(all_scores_expanded, i, j)
    rows.append({"min": i, "max": j, "kappa": results[2]})
thresholded_kappas = pd.DataFrame(rows)
#thresholded_kappas.to_csv("thresholding_results\\thrKappas_jina_title+desc_painpoints.csv", index=False)
thresholded_kappas

C:\Users\ass3352\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:745: RuntimeWarning: invalid value encountered in divide
  expected = np.outer(sum0, sum1) / np.sum(sum0)
C:\Users\ass3352\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:745: RuntimeWarning: invalid value encountered in divide
  expected = np.outer(sum0, sum1) / np.sum(sum0)
C:\Users\ass3352\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:745: RuntimeWarning: invalid value encountered in divide
  expected = np.outer(sum0, sum1) / np.sum(sum0)
C:\Users\ass3352\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:745: RuntimeWarning: invalid value encountered in divide
  expected = np.outer(sum0, sum1) / np.sum(sum0)
C:\Users\ass3352\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:745: RuntimeWarning: invalid value encountered in divide
  expected = np.outer(su

,min,max,kappa
0,0.9,1.0,1.000000
1,0.8,0.9,0.980526
2,0.7,0.8,0.671032
3,0.6,0.7,0.000000
4,0.5,0.6,NaN
5,0.4,0.5,NaN
6,0.3,0.4,NaN
7,0.2,0.3,NaN
8,0.1,0.2,NaN
9,0.0,0.1,NaN
